# Crebto Bot v0.7 — Quiet Wolf (Sei EVM Dry-Run)\n\nThis notebook installs Rust, clones the repo, builds the bot, and runs it on Sei EVM.\n\n**Required Kaggle Secrets** (add these in the Kaggle UI: Add-ons → Secrets):\n| Secret Name | Example Value | Required? |\n|-------------|---------------|-----------|\n| `WS_RPC_URL` | `wss://evm-ws.sei-apis.com` | ✅ Yes |\n| `DRY_RUN` | `true` | ✅ Yes |\n| `DRAGONSWAP_FACTORY` | `0x71f6b49...` | ✅ Yes |\n| `PRIVATE_KEY` | `0xabc...` | ❌ Only for LIVE |\n| `EXECUTOR_ADDRESS` | `0x123...` | ❌ Only for LIVE |\n| `CONTRACT_ADDRESS` | ` 0x456...` | ❌ Only for LIVE |\n\nKaggle kernel stays alive ~9 hours. Use GitHub Actions to restart every 12h.

In [ ]:
# CELL 1: Install Rust toolchain\n!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y\nimport os\nos.environ['PATH'] += ':/root/.cargo/bin'\n!rustc --version && cargo --version

In [ ]:
# CELL 2: Clone repo from GitHub (pulls latest v0.7 code)\n%cd /kaggle/working\n!rm -rf crebto 2>/dev/null; git clone https://github.com/yusef47/crebto.git\n%cd /kaggle/working/crebto/bot

In [ ]:
# CELL 3: Read Kaggle Secrets and write .env file\nfrom kaggle_secrets import UserSecretsClient\n\nclient = UserSecretsClient()\n\ndef get_secret(name, default=''):\n    try:\n        return client.get_secret(name)\n    except:\n        return default\n\nenv_lines = [\n    f"WS_RPC_URL={get_secret('WS_RPC_URL', 'wss://evm-ws.sei-apis.com')}",\n    f"HTTP_RPC_URL={get_secret('HTTP_RPC_URL', 'https://evm-rpc.sei-apis.com')}",\n    f"CHAIN_ID={get_secret('CHAIN_ID', '1329')}",\n    f"DRY_RUN={get_secret('DRY_RUN', 'true')}",\n    f"PRIVATE_KEY={get_secret('PRIVATE_KEY')}",\n    f"EXECUTOR_ADDRESS={get_secret('EXECUTOR_ADDRESS')}",\n    f"CONTRACT_ADDRESS={get_secret('CONTRACT_ADDRESS')}",\n    f"MAX_GAS_PRICE_GWEI={get_secret('MAX_GAS_PRICE_GWEI', '100')}",\n    f"EXECUTION_GAS_LIMIT={get_secret('EXECUTION_GAS_LIMIT', '600000')}",\n    f"MIN_ETH_BALANCE={get_secret('MIN_ETH_BALANCE', '1.0')}",\n    f"MIN_LIQUIDITY_USD={get_secret('MIN_LIQUIDITY_USD', '5000')}",\n    f"MAX_LIQUIDITY_USD={get_secret('MAX_LIQUIDITY_USD', '30000')}",\n    f"MIN_PROFIT_USD={get_secret('MIN_PROFIT_USD', '0.20')}",\n    f"MAX_TAX_BPS={get_secret('MAX_TAX_BPS', '500')}",\n    f"MAX_TRADES_PER_HOUR={get_secret('MAX_TRADES_PER_HOUR', '15')}",\n    f"MAX_DAILY_LOSS_USD={get_secret('MAX_DAILY_LOSS_USD', '10.0')}",\n    f"MAX_LOSS_PER_HOUR_USD={get_secret('MAX_LOSS_PER_HOUR_USD', '5.0')}",\n    f"MAX_CONSECUTIVE_FAILURES={get_secret('MAX_CONSECUTIVE_FAILURES', '50')}",\n    f"DRAGONSWAP_FACTORY={get_secret('DRAGONSWAP_FACTORY', '0x71f6b49ae1558357bbb5a6074f1143c46cbca03d')}",\n    f"SAPPHIRE_FACTORY={get_secret('SAPPHIRE_FACTORY')}",\n    f"SLIPPAGE_BPS={get_secret('SLIPPAGE_BPS', '15')}",\n    f"PROBE_SIZES_USD={get_secret('PROBE_SIZES_USD', '50,100')}",\n]\n\nwith open('/kaggle/working/crebto/bot/.env', 'w') as f:\n    f.write('\n'.join(env_lines))\n\nprint('✅ .env written from Kaggle Secrets')\nprint('Secrets used:', [l.split('=')[1] for l in env_lines if 'DRY_RUN' in l or 'DRAGONSWAP' in l or 'WS_RPC' in l])

In [ ]:
# CELL 4: Build the bot (first build takes ~10-15 min)\n%cd /kaggle/working/crebto/bot\n!cargo build --release 2>&1 | tail -30

In [ ]:
# CELL 5: Run the bot\n# This streams logs until the kernel dies (~9 hours)\n%cd /kaggle/working/crebto/bot\n!cargo run --release 2>&1 | tee /kaggle/working/bot_log.txt

## 📋 What to look for in the output\n\n- `🔍 Discovered X pools with >$5000.00 liquidity` — factory scanner working\n- `✅ Safe pool registered` — pools passing all 7 safety filters\n- `🎯 DRY RUN OPPORTUNITY DETECTED` — simulated arbitrage found\n- `Projected NET PROFIT to Wallet: $X.XX` — estimated profit (simulated)\n\nIf you see `0 safe pools registered`, check that the DragonSwap factory is alive.